# Experiments Results

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from tqdm.auto import tqdm

from swot_toolkit.analysis import open_sites_and_dates
from swot_toolkit.pipe2 import open_output_dir


## Read results

In [3]:
sites_dates = open_sites_and_dates("/data/swot/output")

In [19]:
results = pd.DataFrame()
opera_columns = ["opera_s2", "opera_s1"]

for region, dates in tqdm(sites_dates.items(), desc="Regions"):
    for date in dates:
        base_dir, aoi, _ = open_output_dir(region=region, date=date)

        opera_df = pd.read_parquet(base_dir / f"results/{region}_{date}_OPERA_metrics.parquet")
        swot_df = pd.read_parquet(base_dir / f"results/{region}_{date}_metrics.parquet")
        metrics_df = pd.concat([swot_df[["A2B2"]], opera_df], axis=1)
        metrics_df["Region"] = region
        metrics_df["Date"] = pd.to_datetime(date)
        results = pd.concat([results, metrics_df], axis=0)

results = results.rename(
    columns={"A2B2": "SWOT", "opera_s2": "OPERA S2", "opera_s1": "OPERA S1"}
).drop(columns=["opera_s2 incl. partial", "opera_s1 incl. partial"])

results = results.reset_index(names=["Metric"]).set_index(["Region", "Date", "Metric"])
results

Regions:   0%|          | 0/5 [00:00<?, ?it/s]

SWOT  OPERA S2  OPERA S1
Region      Date       Metric                               
Curua-Una   2024-07-13 iou        0.7567    0.9415       NaN
                       f1         0.8615    0.9699       NaN
                       precision  0.8759    0.9967       NaN
                       recall     0.8476    0.9445       NaN
            2025-08-14 iou        0.8052    0.8985    0.8607
                       f1         0.8921    0.9465    0.9252
                       precision  0.9370    0.9963    0.9974
                       recall     0.8512    0.9015    0.8627
Northeast   2024-05-29 iou        0.6224    0.8106       NaN
                       f1         0.7673    0.8954       NaN
                       precision  0.8862    0.9788       NaN
                       recall     0.6765    0.8251       NaN
            2025-07-20 iou        0.7707    0.8573    0.6738
                       f1         0.8705    0.9232    0.8051
                       precision  0.8957    0.9518    0.9896
                       recall     0.8467    0.8962    0.6786
Rio_Branco  2024-04-03 iou        0.2658    0.6474       NaN
                       f1         0.4200    0.7859       NaN
                       precision  0.4357    0.9976       NaN
                       recall     0.4053    0.6484       NaN
            2025-09-07 iou        0.5583    0.7739    0.6903
                       f1         0.7166    0.8725    0.8168
                       precision  0.7823    0.9982    0.9941
                       recall     0.6610    0.7750    0.6931
Rio_Madeira 2024-08-21 iou        0.8404    0.9256       NaN
                       f1         0.9133    0.9614       NaN
                       precision  0.9137    0.9971       NaN
                       recall     0.9128    0.9281       NaN
            2025-07-21 iou        0.7783    0.9297    0.9020
                       f1         0.8753    0.9636    0.9485
                       precision  0.8443    0.9974    0.9940
                       recall     0.9087    0.9320    0.9069
Rio_Negro   2024-11-29 iou        0.5636    0.8789       NaN
                       f1         0.7209    0.9355       NaN
                       precision  0.5988    0.9996       NaN
                       recall     0.9054    0.8792       NaN
            2025-08-07 iou        0.7832    0.9474    0.8713
                       f1         0.8784    0.9730    0.9313
                       precision  0.8904    0.9964    0.9998
                       recall     0.8668    0.9506    0.8715

## Calculate Overall Median

In [20]:
# To calculate median over all regions and dates, we group by Metric
overall_results = results.groupby(level=["Metric"]).median().sort_index().round(3)
overall_results

,SWOT,OPERA S2,OPERA S1
Metric,,,
f1,0.866,0.941,0.925
iou,0.764,0.889,0.861
precision,0.881,0.997,0.994
recall,0.849,0.899,0.863


In [39]:
import plotly.graph_objects as go

# Rename regions for display
rename_metrics = {
    "f1": "F1 Score",
    "iou": "IoU",
    "precision": "Precision",
    "recall": "Recall",
}

# Define colors for each metric (colorblind-friendly palette)
colors = {
    "SWOT": "#0987E7",  # Blue
    "OPERA S2": "#0CC228",  # Orange
    "OPERA S1": "#967024",  # Green
}

plot_data = overall_results.copy()
plot_data.index = plot_data.index.map(rename_metrics)

fig = go.Figure()

for sensor in plot_data.columns:
    fig.add_trace(
        go.Bar(
            name=sensor,
            x=plot_data.index,
            y=plot_data.loc[:, sensor],
            textposition="auto",
            width=0.2,
            marker_color=colors[sensor]
        )
    )

fig.update_layout(
    barmode="group",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font={"family": "Arial, sans-serif", "size": 16, "color": "black"},
    legend={
        "title": {"text": "", "font": {"size": 20}},
        "x": 1.02,
        "y": 1,
        "xanchor": "left",
        "yanchor": "top",
        "bgcolor": "rgba(255, 255, 255, 0.8)",
        "bordercolor": "black",
        "borderwidth": 1,
    },
    width=900,
    height=500,
    margin={"t": 80, "b": 120, "l": 80, "r": 40},
)
fig.update_xaxes(
    showgrid=False,
    showline=True,
    linewidth=2,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=1.5,
    tickcolor="black",
    tickangle=-45,
    tickfont={"size": 18},
)

fig.update_yaxes(
    range=[0, 1.05],
    showgrid=False,
    showline=True,
    linewidth=2,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=1.5,
    tickcolor="black",
    tickmode="linear",
    tick0=0.0,
    dtick=0.2,
    tickfont={"size": 18},
)


In [23]:
plot_data

,SWOT,OPERA S2,OPERA S1
Metric,,,
NaN,0.866,0.941,0.925
NaN,0.764,0.889,0.861
NaN,0.881,0.997,0.994
NaN,0.849,0.899,0.863
